# Transformer preparation | GPU, tokenization, and memory validation

**Scope.** This notebook validates the Kaggle runtime, project data integrity, split isolation, multilingual tokenizer revisions, token-length distributions, and GPU memory feasibility. It does not perform task-specific model fine-tuning or report predictive performance.

**Runtime.** Kaggle with a CUDA-enabled NVIDIA GPU; Internet access is needed for the initial public tokenizer and model downloads. All dataset texts are processed locally. Tokenizers operate on CPU; the synthetic GPU memory probes load model weights temporarily but use no project documents.

**Inputs.** `training_NLP.zip`, `test_NLP.zip`, and the project setup kit (`nlp_setup.py` with accompanying reference files), either extracted or packaged as `kit_setup_nlp.zip`.

**Outputs.** Audit files, runtime metadata, tokenizer snapshots, token-length statistics, and synthetic memory-probe measurements under `/kaggle/working/nlp_setup/`. Saved artifacts may be reused in later sessions; reproducible execution requires the same input fingerprints and model revisions.

**Execution order.** Run sections 1–10 sequentially, then the optional GPU memory probes in section 11. For artifact inspection without downloads or GPU compute, read the exported JSON/CSV files from a previously saved `preparazione_transformer.zip`. Older notebook outputs are retained as execution records; they are not evidence that unexecuted sections completed.


## 1. CUDA runtime verification

Verify PyTorch, CUDA, and visible GPU memory before running GPU-dependent checks. A small matrix multiplication confirms basic CUDA operation, not sufficient memory for model training.

In [1]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Runtime CUDA:', torch.version.cuda)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable: enable a GPU accelerator and restart the session if required.')
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {props.name} | total memory {props.total_memory / 1024**3:.2f} GiB')
with torch.no_grad():
    a = torch.ones((32, 32), device='cuda:0')
    b = a @ a
    torch.cuda.synchronize()
    assert torch.all(b == 32).item()
del a, b
torch.cuda.empty_cache()
print('GPU_OK — CUDA computation passed; no model loaded.')


PyTorch: 2.11.0+cu130
CUDA disponibile: True
Runtime CUDA: 13.0
GPU 0: Tesla T4 | memoria totale 14.56 GiB
GPU 1: Tesla T4 | memoria totale 14.56 GiB
GPU_OK — piccolo calcolo CUDA riuscito. Nessun modello caricato.


## 2. Input and output paths

Paths default to Kaggle directories and can be overridden through environment variables. Source overrides must point to an archive or a dataset directory containing `training/` or `test/`, not a language subdirectory.

In [2]:
from pathlib import Path
import os
import sys
import json
import zipfile
import shutil

INPUT_ROOT = Path(os.environ.get('NLP_INPUT_ROOT', '/kaggle/input'))
OUTPUT_ROOT = Path(os.environ.get('NLP_OUTPUT_ROOT', '/kaggle/working/nlp_setup'))
TRAIN_OVERRIDE = os.environ.get('NLP_TRAIN_SOURCE') or None
TEST_OVERRIDE = os.environ.get('NLP_TEST_SOURCE') or None
KIT_OVERRIDE = os.environ.get('NLP_KIT_DIR') or None
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('Input:', INPUT_ROOT)
print('Output:', OUTPUT_ROOT)


Input: /kaggle/input
Output: /kaggle/working/nlp_setup


## 3. Project setup kit and taxonomy

Locate an extracted setup kit or safely extract its ZIP archive. Reference inputs remain unchanged; generated files are written under the output directory.

In [3]:
if KIT_OVERRIDE:
    KIT_DIR = Path(KIT_OVERRIDE)
else:
    found = sorted({p.parent for p in INPUT_ROOT.rglob('nlp_setup.py')})
    if len(found) == 1:
        KIT_DIR = found[0]
    elif len(found) > 1:
        raise ValueError('Multiple setup kits found; set NLP_KIT_DIR explicitly.')
    else:
        archives = list(INPUT_ROOT.rglob('kit_setup_nlp.zip'))
        if len(archives) != 1:
            raise ValueError('Exactly one setup kit ZIP is required when no extracted kit is available.')
        KIT_DIR = OUTPUT_ROOT / '_kit'
        KIT_DIR.mkdir(exist_ok=True)
        with zipfile.ZipFile(archives[0]) as z:
            for info in z.infolist():
                target = (KIT_DIR / info.filename).resolve()
                if not target.is_relative_to(KIT_DIR.resolve()):
                    raise ValueError('Unsafe path in setup kit ZIP.')
            z.extractall(KIT_DIR)

sys.path.insert(0, str(KIT_DIR))
import nlp_setup as setup
import numpy as np
import pandas as pd
from IPython.display import display

taxonomy = json.loads((KIT_DIR / 'tassonomia_progetto_v1.json').read_text(encoding='utf-8'))
setup.require(len(taxonomy['coarse']) == 22 and len(taxonomy['fine']) == 96, 'Unexpected taxonomy dimensions.')
setup.require(len(set(taxonomy['fine'])) == 96, 'Duplicate fine-grained classes.')
setup.require(all(l == 'Other' or l.rsplit(': ', 1)[0] in taxonomy['coarse'] for l in taxonomy['fine']),
              'Fine-grained taxonomy entry has no valid parent.')
environment = setup.environment_info()
setup.save_json(OUTPUT_ROOT / 'environment.json', environment)
print('Kit:', KIT_DIR)
print('Taxonomy: 22 coarse / 96 fine. Environment:')
display(environment)


Kit: /kaggle/input/datasets/nizarmabrouk/nlp-semeval-2025-dati-e-setup/kit_setup_nlp
Classi: 22 coarse / 96 fine. Ambiente:


{'python': '3.13.16',
 'platform': 'Linux-6.18.55+-x86_64-with-glibc2.39',
 'packages': {'numpy': '2.1.3',
  'pandas': '2.3.3',
  'scikit-learn': '1.6.1',
  'torch': '2.11.0+cu130',
  'transformers': '5.20.2',
  'tokenizers': '0.23.3'},
 'gpu': ['Tesla T4', 'Tesla T4'],
 'cuda_runtime': '13.0'}

## 4. Training records and test-text integrity

Validate training annotations against the project taxonomy and recorded fingerprints. Repeated labels within a document are deduplicated as multi-label targets. Test documents are loaded as text only; test labels are not accessed.

In [4]:
training_source = setup.locate_source(INPUT_ROOT, 'training_NLP.zip', 'training', TRAIN_OVERRIDE)
test_source = setup.locate_source(INPUT_ROOT, 'test_NLP.zip', 'test', TEST_OVERRIDE)
train_records = setup.load_training(training_source, taxonomy)
course_texts = setup.load_course_texts(test_source)
expected = json.loads((KIT_DIR / 'expected_data_fingerprints.json').read_text(encoding='utf-8'))
setup.require(setup.fingerprint_records(train_records) == expected['training_records'],
              'Training fingerprint does not match the verified reference.')
setup.require(setup.fingerprint_records(course_texts) == expected['course_texts'],
              'Test-text fingerprint does not match the verified reference.')

data_counts = pd.DataFrame([
    {'language': lang,
     'training': sum(r['language'] == lang for r in train_records),
     'test_text_only': sum(r['language'] == lang for r in course_texts)}
    for lang in setup.LANGUAGES
])
display(data_counts)
repeated_docs = sum(r['repeated_labels_removed'] > 0 for r in train_records)
print('Training documents with duplicate labels removed:', repeated_docs)


,lingua,training,test_solo_testi
0,EN,399,41
1,BG,401,35
2,HI,366,35
3,PT,400,35
4,RU,133,32


Documenti training con ripetizioni di etichette deduplicate: 549


## 5. Lexical near-duplicate screening

Use within-language cosine similarity of word-trigram TF-IDF vectors at a threshold of 0.80, plus whitespace-normalized exact matches across languages. Similarity candidates are grouped, and auxiliary training texts connected to test texts are excluded. This conservative lexical screen cannot identify all paraphrases or translations. Audit features are not used as classifier features.

In [5]:
similarity = setup.audit_similarity(train_records, course_texts, threshold=0.80)
reference = json.loads((KIT_DIR / 'near_duplicate_audit_reference.json').read_text(encoding='utf-8'))
pair_keys = lambda a: {(p['a'], p['b']) for p in a['pairs']}
setup.require(pair_keys(similarity) == pair_keys(reference),
              'Near-duplicate candidates differ from the verified reference.')
setup.require(similarity['excluded_training_keys'] == reference['excluded_training_keys'],
              'Training exclusions differ from the verified reference.')
setup.save_json(OUTPUT_ROOT / 'near_duplicate_audit.json', similarity)
pd.DataFrame(similarity['pairs']).to_csv(OUTPUT_ROOT / 'similarity_pairs.csv', index=False)
print('Candidate pairs:', len(similarity['pairs']))
print('Excluded auxiliary documents:', len(similarity['excluded_training_keys']))
display(pd.DataFrame({'excluded_document': similarity['excluded_training_keys']}))


Coppie candidate: 74
Documenti ausiliari esclusi: 11


,documento_escluso
0,train/BG/A9_BG_5548.txt
1,train/HI/HI_115.txt
2,train/HI/HI_161.txt
3,train/HI/HI_177.txt
4,train/HI/HI_23.txt
5,train/HI/HI_56.txt
6,train/HI/HI_81.txt
7,train/HI/HI_99.txt
8,train/PT/PT_105.txt
9,train/PT/PT_115.txt


## 6. Split manifest validation

Validate split version `v2` and document-group isolation. Two English records that differ only in quotation marks (`EN_CC_200107.txt`, `EN_CC_200093.txt`) belong to the same group. All 399 English records remain represented across 395 isolation groups; no labels or predictions are used to select folds.

In [6]:
manifest = setup.validate_manifest(train_records, similarity, KIT_DIR / 'split_manifest_v2.csv')
split_counts = manifest.groupby(['fold', 'partition']).size().unstack()
display(split_counts[['inner_train', 'inner_validation', 'outer_evaluation']])
manifest.to_csv(OUTPUT_ROOT / 'verified_split_manifest_v2.csv', index=False)
print('SPLIT_OK — 399 English records; each evaluated once per seed across three folds.')
print('SPLIT_OK — linked English records do not cross fold partitions.')


partition,inner_train,inner_validation,outer_evaluation
fold,,,
1,213,53,133
2,212,53,134
3,214,53,132


OK: 399 documenti EN; ogni documento valutato una sola volta per seed nei tre turni.
OK: gruppi identici e candidati inglesi non attraversano partizioni nello stesso turno.


## 7. Fold setup without model fitting

Build a reference fold and its training target matrices. Outer-evaluation targets remain outside the development workflow. Seeding improves repeatability but does not guarantee identical outcomes across hardware and package versions.

In [7]:
FOLD = 1
TRAIN_SEED = 42
MULTILINGUAL = False
setup.seed_everything(TRAIN_SEED)
fold_data = setup.make_fold(train_records, manifest, similarity, FOLD, MULTILINGUAL)
y_train_coarse = setup.binary_labels([r['coarse'] for r in fold_data['inner_train']], taxonomy['coarse'])
y_train_fine = setup.binary_labels([r['fine'] for r in fold_data['inner_train']], taxonomy['fine'])
print('Fold:', FOLD, '| Multilingual:', MULTILINGUAL)
print({role: len(rows) for role, rows in fold_data.items()})
print('Training target shapes:', y_train_coarse.shape, y_train_fine.shape)
print('Seeding improves reproducibility but cannot guarantee identical results across environments.')


Fold: 1 | Multilingue: False
{'inner_train': 213, 'inner_validation': 53, 'outer_evaluation': 133}
Target di training: (213, 22) (213, 96)
Il seed migliora la riproducibilità, ma non garantisce identità tra hardware e versioni diverse.


## 8. Evaluation-function consistency

Compare the strict project sample-F1 implementation with the original version-3 scorer on synthetic labels restricted to their shared class set. The original scorer silently ignores unknown labels; the strict wrapper rejects them. No held-out labels are used.

In [8]:
import importlib.util
spec = importlib.util.spec_from_file_location('official_st2_scorer', KIT_DIR / 'official_v3/subtask2_scorer.py')
official = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official)
rng = np.random.default_rng(20261103)
for level, filename in [('coarse', 'subtask2_narratives.txt'), ('fine', 'subtask2_subnarratives.txt')]:
    classes = (KIT_DIR / 'official_v3' / filename).read_text(encoding='utf-8').splitlines()
    gold = [[], [classes[0]], [classes[0]], [classes[0], classes[1]]]
    pred = [[], [classes[0]], [], [classes[1]]]
    for _ in range(30):
        gold.append(rng.choice(classes, size=int(rng.integers(0, 5)), replace=False).tolist())
        pred.append(rng.choice(classes, size=int(rng.integers(0, 5)), replace=False).tolist())
    ours = setup.sample_f1_strict(gold, pred, classes)
    theirs = official.evaluate(gold, pred, classes)
    setup.require(np.allclose([ours['mean'], ours['document_std']], theirs, atol=1e-12),
                  'Scorers disagree on synthetic examples.')
print('SCORER_OK — strict scorer matches the official v3 scorer on common synthetic labels.')
try:
    setup.sample_f1_strict([['UNKNOWN_LABEL']], [[]], taxonomy['fine'])
except ValueError:
    print('SCORER_OK — unknown labels are rejected.')
else:
    raise AssertionError('Unknown label was not rejected.')


OK: metrica coerente con scorer v3 sui casi sintetici a classi comuni.
OK: le etichette sconosciute bloccano il calcolo.


## 9. Validated preparation summary

Export input counts, split sizes, environment details, and known limitations to `setup_summary.json`. The run-directory utility is available for later experiments, but no model is fitted here.

In [9]:
fold_sizes = {}
for fold in (1, 2, 3):
    fold_sizes[str(fold)] = {}
    for multi in (False, True):
        parts = setup.make_fold(train_records, manifest, similarity, fold, multi)
        fold_sizes[str(fold)]['multilingual' if multi else 'EN_only'] = {k:len(v) for k,v in parts.items()}
summary = {
    'status': 'SETUP_OK', 'split_version': 'v2', 'taxonomy_version': 'project_v1_96fine',
    'training_documents_original': len(train_records),
    'course_test_texts': len(course_texts), 'course_test_labels_loaded': False,
    'coarse_classes': len(taxonomy['coarse']), 'fine_classes': len(taxonomy['fine']),
    'similarity_candidate_pairs': len(similarity['pairs']),
    'auxiliary_documents_excluded': len(similarity['excluded_training_keys']),
    'documents_with_repeated_labels': repeated_docs,
    'fold_sizes': fold_sizes, 'environment': environment,
    'limitations': ['Lexical screen does not rule out translations or all paraphrases.',
                    'Course test historically consulted; not an untouched benchmark.',
                    'Extended taxonomy is a project definition, not an official new release.'],
    'training_started': False
}
setup.save_json(OUTPUT_ROOT / 'setup_summary.json', summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print('Preparation summary saved:', OUTPUT_ROOT / 'setup_summary.json')


{
  "status": "SETUP_OK",
  "split_version": "v2",
  "taxonomy_version": "progetto_v1_96fine",
  "training_documents_original": 1699,
  "course_test_texts": 178,
  "course_test_labels_loaded": false,
  "coarse_classes": 22,
  "fine_classes": 96,
  "similarity_candidate_pairs": 74,
  "auxiliary_documents_excluded": 11,
  "documents_with_repeated_labels": 549,
  "fold_sizes": {
    "1": {
      "EN_only": {
        "inner_train": 213,
        "inner_validation": 53,
        "outer_evaluation": 133
      },
      "multilingual": {
        "inner_train": 1502,
        "inner_validation": 53,
        "outer_evaluation": 133
      }
    },
    "2": {
      "EN_only": {
        "inner_train": 212,
        "inner_validation": 53,
        "outer_evaluation": 134
      },
      "multilingual": {
        "inner_train": 1501,
        "inner_validation": 53,
        "outer_evaluation": 134
      }
    },
    "3": {
      "EN_only": {
        "inner_train": 214,
        "inner_validation": 53,
     

## 10. Tokenizer snapshots and repository revisions

Download and pin the fast tokenizers for [multilingual BERT](https://huggingface.co/google-bert/bert-base-multilingual-cased) and [XLM-RoBERTa](https://huggingface.co/FacebookAI/xlm-roberta-base). Store repository commit hashes and SHA-256 digests of the saved tokenizer files. Public repositories require no personal access tokens. This stage does not download model weights.

In [ ]:
from transformers import AutoTokenizer
from huggingface_hub import HfApi
import hashlib

TOKEN_DIR = OUTPUT_ROOT / 'tokenizer_audit'
TOKEN_DIR.mkdir(parents=True, exist_ok=True)
MODEL_IDS = {'mBERT':'google-bert/bert-base-multilingual-cased',
             'XLM-R':'FacebookAI/xlm-roberta-base'}
tokenizers = {}; tokenizer_metadata = {}
for name, model_id in MODEL_IDS.items():
    print('Loading tokenizer:', name, flush=True)
    revision = HfApi().model_info(model_id).sha
    tok = AutoTokenizer.from_pretrained(model_id, revision=revision, use_fast=True, trust_remote_code=False)
    if not tok.is_fast:
        raise RuntimeError('Expected a fast tokenizer; inspect package versions and snapshot.')
    dest = TOKEN_DIR / name
    tok.save_pretrained(dest)
    tokenizer_metadata[name] = {'model_id':model_id,'revision':revision,
        'tokenizer_class':type(tok).__name__,'is_fast':tok.is_fast,
        'model_max_length':int(tok.model_max_length),
        'special_tokens_single_sequence':int(tok.num_special_tokens_to_add(pair=False)),
        'files_sha256':{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in dest.iterdir() if p.is_file()}}
    tokenizers[name] = tok
setup.save_json(TOKEN_DIR / 'tokenizer_metadata.json', tokenizer_metadata)
print(json.dumps(tokenizer_metadata,ensure_ascii=False,indent=2))
print('TOKENIZERS_OK — tokenizer snapshots saved; no model weights loaded.')


### Token-length and truncation audit

Measure untruncated content token counts for eligible training documents. For sequence limits of 128, 256, 384, and 512, reserve capacity for special tokens and estimate truncation frequency and token loss by language. Token-loss estimates do not measure lost task-relevant information or rank long-document strategies.

In [ ]:
active_records = sorted([r for r in train_records if r['key'] not in set(similarity['excluded_training_keys'])],
                        key=lambda r:r['key'])
setup.require(len(active_records)==1688,'Unexpected eligible training record count.')
length_rows = []; truncation_rows = []
for name, tok in tokenizers.items():
    print('Tokenizing:', name, flush=True)
    lengths = []
    for start in range(0,len(active_records),32):
        batch=active_records[start:start+32]
        encoded=tok([r['text'] for r in batch],add_special_tokens=False,
                    truncation=False,padding=False,return_attention_mask=False,
                    return_token_type_ids=False,verbose=False)
        lengths.extend(len(ids) for ids in encoded['input_ids'])
    special=tok.num_special_tokens_to_add(pair=False)
    for record,n in zip(active_records,lengths):
        length_rows.append({'model':name,'language':record['language'],
            'document_id':record['document_id'],'content_tokens':n,
            'total_tokens_with_special':n+special})
    for language in setup.LANGUAGES:
        n=np.array([length for r,length in zip(active_records,lengths) if r['language']==language])
        for limit in (128,256,384,512):
            setup.require(limit<=tok.model_max_length,'Sequence length exceeds tokenizer model limit.')
            capacity=limit-special
            lost=np.maximum(n-capacity,0)
            fractions=np.divide(lost,n,out=np.zeros(len(n),dtype=float),where=n>0)
            truncation_rows.append({'model':name,'language':language,'documents':len(n),
                'max_length':limit,'content_capacity':capacity,
                'median_total_tokens':float(np.median(n+special)),
                'p90_total_tokens':float(np.quantile(n+special,.9)),
                'truncated_documents_pct':float(100*np.mean(lost>0)),
                'mean_content_tokens_lost_pct':float(100*fractions.mean()),
                'pooled_content_tokens_lost_pct':float(100*lost.sum()/n.sum())})
length_table=pd.DataFrame(length_rows)
truncation_table=pd.DataFrame(truncation_rows)
length_table.to_csv(TOKEN_DIR/'document_token_lengths.csv',index=False)
truncation_table.to_csv(TOKEN_DIR/'truncation_by_language.csv',index=False)
display(truncation_table.loc[truncation_table.language=='EN',
    ['model','max_length','documents','median_total_tokens','p90_total_tokens',
     'truncated_documents_pct','mean_content_tokens_lost_pct']])
print('TOKEN_LENGTHS_OK — token counts saved by tokenizer and language.')


### Tokenizer audit export

Save `tokenizer_summary.json` and a ZIP archive of the tokenizer audit. The English-language truncation table is included for method selection. Tokenizer measurements do not establish whether model training fits in available GPU memory.

In [ ]:
tokenizer_summary={'status':'TOKENIZER_AUDIT_OK','training_documents':len(active_records),
    'models':tokenizer_metadata,'environment':environment,
    'english_truncation':truncation_table[truncation_table.language=='EN'].to_dict('records'),
    'course_test_labels_loaded':False,'model_weights_loaded':False,'training_started':False}
setup.save_json(TOKEN_DIR/'tokenizer_summary.json',tokenizer_summary)
archive=shutil.make_archive(str(TOKEN_DIR),'zip',TOKEN_DIR)
print(json.dumps(tokenizer_summary,ensure_ascii=False,indent=2))
print('Tokenizer archive:',archive)


## 11. Synthetic GPU memory probes (optional)

The probes load the **model weights** at pinned revisions and perform three optimization steps on synthetic inputs using a 118-label temporary classification head (22 coarse + 96 fine labels). They measure runtime feasibility for the stated configuration only, not training quality or task metrics. These cells require Internet for uncached weights and a CUDA GPU. Execute both probes sequentially; the first probe must release GPU resources before the second begins. Failed or incomplete probes must not be interpreted as successful benchmarks.

**Recovery.** Previously saved `gpu_memory_probe/*_probe.json` files can be inspected without rerunning the probes. The final archive cell requires both tokenizer and probe directories; skip it when only tokenizer results are available.

In [ ]:
# Synthetic inputs only; no dataset examples or evaluation metrics.
import gc
import json
import time
from pathlib import Path
import torch
from transformers import AutoModelForSequenceClassification

PROBE_MODELS = {
    'mBERT': ('google-bert/bert-base-multilingual-cased',
              '3f076fdb1ab68d5b2880cb87a0886f315b8146f8'),
    'XLM-R': ('FacebookAI/xlm-roberta-base',
              'e73636d4f797dec63c3081bb6ed5c7b0bb3f2089'),
}

def _memory_probe(name):
    model_id, revision = PROBE_MODELS[name]
    torch.manual_seed(42)
    torch.cuda.manual_seed_all(42)
    device = torch.device('cuda:0')
    print(f'Loading model weights: {name}.', flush=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, revision=revision, trust_remote_code=False,
        num_labels=118, problem_type='multi_label_classification',
        attn_implementation='eager',
    ).float().to(device)
    model.train()
    # Conservative configuration: eager attention without gradient checkpointing.
    # Temporary 22+96 classifier head; not the final experimental architecture.
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, foreach=False)
    scaler = torch.amp.GradScaler('cuda', init_scale=1024.0)
    ids = torch.full((2, 512), 100, dtype=torch.long, device=device)
    mask = torch.ones_like(ids)
    targets = torch.zeros((2, 118), device=device)
    targets[:, 0] = 1
    targets[:, 22] = 1
    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start = time.perf_counter()
    losses = []
    successful_steps = 0
    # Measure both optimizer state initialization and subsequent steps.
    for step in range(3):
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss = torch.nn.functional.binary_cross_entropy_with_logits(logits.float(), targets)
        if not torch.isfinite(loss).item():
            raise RuntimeError('Non-finite loss during synthetic memory probe.')
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        finite = bool(torch.isfinite(norm).item())
        scaler.step(optimizer)
        scaler.update()
        successful_steps += int(finite)
        losses.append(float(loss.detach().cpu()))
        print(f'Synthetic step {step + 1}/3: finite gradients={finite}', flush=True)
        del logits, loss, norm
    torch.cuda.synchronize(device)
    if successful_steps != 3 or not optimizer.state:
        raise RuntimeError('Incomplete memory probe: one or more optimization steps were skipped.')
    free_bytes, total_bytes = torch.cuda.mem_get_info(device)
    return {
        'status': 'GPU_MEMORY_PROBE_OK', 'model': name,
        'model_id': model_id, 'revision': revision,
        'gpu': torch.cuda.get_device_name(device), 'torch': torch.__version__,
        'batch_size': 2, 'sequence_length': 512, 'output_labels': 118,
        'precision': 'FP32 parameters + FP16 autocast + GradScaler',
        'attention': 'eager', 'gradient_checkpointing': False,
        'optimizer': 'AdamW, foreach=False', 'successful_steps': successful_steps,
        'peak_allocated_GiB': torch.cuda.max_memory_allocated(device) / 1024**3,
        'peak_reserved_GiB': torch.cuda.max_memory_reserved(device) / 1024**3,
        'free_device_GiB_at_end': free_bytes / 1024**3,
        'total_device_GiB': total_bytes / 1024**3,
        'three_steps_seconds': time.perf_counter() - start,
        'synthetic_losses_not_metrics': losses,
        'real_documents_used': False, 'checkpoint_saved': False,
        'note': 'Technical probe only; model discarded. Repeat if training recipe changes.',
    }

def run_memory_probe(name):
    if name not in PROBE_MODELS:
        raise ValueError(f'Unknown model: {name}')
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable; enable a Kaggle GPU accelerator.')
    gc.collect()
    torch.cuda.empty_cache()
    try:
        result = _memory_probe(name)
    finally:
        gc.collect()
        torch.cuda.empty_cache()
    dest = OUTPUT_ROOT / 'gpu_memory_probe'
    dest.mkdir(parents=True, exist_ok=True)
    path = dest / f'{name}_probe.json'
    path.write_text(json.dumps(result, indent=2), encoding='utf-8')
    print(json.dumps(result, indent=2))
    print('Probe saved:', path)
    return result

# Run mBERT first; XLM-R is measured separately.
mbert_probe = run_memory_probe('mBERT')

In [ ]:
xlmr_probe = run_memory_probe('XLM-R')

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

root = OUTPUT_ROOT
folders = [root / "tokenizer_audit", root / "gpu_memory_probe"]

for folder in folders:
    assert folder.is_dir(), f"Missing directory: {folder}"

archive = root / "preparazione_transformer.zip"

with ZipFile(archive, "w", ZIP_DEFLATED) as z:
    for folder in folders:
        for file in sorted(folder.rglob("*")):
            if file.is_file():
                z.write(file, file.relative_to(root))

print("TRANSFORMER_PREPARATION_OK")
print("Preparation archive:", archive)